In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.11.0


In [ ]:
import torch

#上下文向量
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [4]:
#第二个输入词元作为查询向量
query=inputs[1] 

#计算注意力分数
attn_scores_2=torch.empty(inputs.shape[0])
for i,x_i in enumerate(inputs):
    attn_scores_2[i] =torch.dot(x_i,query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [ ]:
attn_weights_2_tmp=attn_scores_2/attn_scores_2.sum()

print("Attention weights:",attn_weights_2_tmp) #注意力权重
print("Sum:",attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


In [6]:
def softmatx_naive(x):
    return torch.exp(x)/torch.exp(x).sum(dim=0)

attn_weights_2_naive=softmatx_naive(attn_scores_2)

print("Attention weights:",attn_weights_2_naive)
print("Sum:",attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [7]:
attn_weights_2=torch.softmax(attn_scores_2,dim=0)

print("Attention weights:",attn_weights_2)
print("Sum:",attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [10]:
#第二个输入词元作为查询向量
query=inputs[1]

context_vec_2=torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2+=attn_weights_2[i]*x_i #上下文向量+=词元嵌入*注意力权重
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


In [12]:
#计算所有的注意力分数 
#注意力分数=词元嵌入*词元嵌入
attn_scores=torch.empty(6,6)
for i,x_i in enumerate(inputs):
    for j,x_j in enumerate(inputs):
        attn_scores[i][j]=torch.dot(x_i,x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [13]:
#@矩阵乘法更快
attn_scores=inputs@inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [ ]:
#归一化
attn_weights=torch.softmax(attn_scores,dim=-1)
print(attn_weights) #注意力权重

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [16]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)

print("All row sums:", attn_weights.sum(dim=-1))

Row 2 sum: 1.0
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [19]:
all_context_vecs=attn_weights@inputs
print(all_context_vecs) #上下文向量

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [20]:
print("Previous 2nd context vector:",context_vec_2)

Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


In [28]:
#3.4.1 逐步计算注意力权重
x_2=inputs[1] #第二个输入元素
d_in=inputs.shape[1] #输入嵌入维度d_in=3
d_out=2 #输出嵌入维度d_out=2

torch.manual_seed(123)

#初始化3个权重矩阵W_q, W_k, W_v:
W_query=torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)
W_key=torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)
W_value=torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)

In [29]:
query_2=x_2@W_query
key_2=x_2@W_key
value_2=x_2@W_value

print(query_2)

tensor([0.4306, 1.4551])


In [30]:
keys=inputs@W_key
values=inputs@W_value

print("keys.shape:",keys.shape)
print("values.shape:",values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


In [31]:
keys_2=keys[1]
attn_scores_22=query_2.dot(keys_2)
print(attn_scores_22)

tensor(1.8524)


In [ ]:
attn_scores_2=query_2@keys.T #第二个元素的注意力分数
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [ ]:
d_k=keys.shape[-1] #2
attn_weights_2=torch.softmax(attn_scores_2/d_k**0.5,dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [35]:
context_vec_2=attn_weights_2@values
print(context_vec_2)

tensor([0.3061, 0.8210])
